# 🩺 Laporan Proyek Machine Learning: Prediksi Risiko Serangan Jantung

* **Metodologi Penelitian:** CRISP-DM 📊
* **Penyusun:** Suci Oktavia Ramadhani & Salsabila Anggraina Putri 👥

---

## 📌 1. Project Overview
Penyakit kardiovaskular, khususnya serangan jantung, tetap menjadi salah satu penyebab utama mortalitas tertinggi di Indonesia. Sifat gejalanya yang acap kali asimtomatik (*silent killer*) diperparah oleh banyaknya faktor risiko multidimensional yang saling berkorelasi. 💔

Proyek ini dicanangkan untuk membangun sebuah sistem komputasi cerdas berbasis *Machine Learning* yang mampu memprediksi tingkat risiko serangan jantung pada masyarakat secara dini. 🩺✨

---

## 🎯 2. Business Understanding

### ❓ Problem Statements
1. Bagaimana karakteristik demografi serta parameter klinis harian dapat memengaruhi kecenderungan risiko serangan jantung seseorang? 📊
2. Apakah implementasi algoritma *Advanced Ensemble Learning* (**XGBoost Classifier**) mampu menghasilkan performa klasifikasi yang lebih superior dibandingkan model standar (**Logistic Regression**)? 🧠

### 🚀 Goals
1. Mengidentifikasi dan menganalisis korelasi prediktif dari berbagai fitur kesehatan terhadap indikasi risiko penyakit jantung. 🔍
2. Mengembangkan model prediktif dengan tingkat sensitivitas (**Recall**) yang optimal demi meminimalisir *False Negative*. 🎯

### 💡 Solution Statement
* **Model Baseline:** *Logistic Regression* 📉
* **Model Advanced:** *XGBoost Classifier* ⚡

---

## 📊 3. Data Understanding
Dataset dari [Kaggle](https://www.kaggle.com/datasets/ankushpanday2/heart-attack-prediction-in-indonesia) memiliki **158.355 baris** dan **28 kolom**. 🗂️

| Fitur | Deskripsi |
|---|---|
| 🧓 `age` | Usia biologis pasien |
| ⚧ `gender` | Jenis kelamin (Male, Female) |
| 🩸 `hypertension` | Riwayat tekanan darah tinggi (0/1) |
| 🍬 `diabetes` | Riwayat diabetes (0/1) |
| 🍟 `cholesterol_level` | Kadar kolesterol (mg/dL) |
| 🍔 `obesity` | Status obesitas (0/1) |
| 🚬 `smoking_status` | Kebiasaan merokok |
| 🥦 `dietary_habits` | Pola makan harian |
| 🤯 `stress_level` | Skala stres (1-10) |
| 🎯 `heart_attack` | **Target** (0 = Rendah, 1 = Tinggi) |

### ▶️ Install dan Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

### ▶️ Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
df = pd.read_csv('/content/drive/MyDrive/ML/heart_attack_prediction_indonesia.csv')

print('=== DIMENSI DATA ===')
print(f'Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}\n')

print('=== INFORMASI DATASET ===')
df.info()

---

## 🔎 4. Exploratory Data Analysis (EDA)

Sebelum melakukan pembersihan, kita bedah terlebih dahulu kondisi data mentah. 🔍

### 😖 Kondisi Data Mentah
- ✅ **Tidak ada data duplikat**
- ⚠️ **Missing Values** — kolom `alcohol_consumption` memiliki **94.848 nilai kosong** (~59.9%)
- 📊 **Distribusi Target** — tidak seimbang: **94.854 (0)** vs **63.501 (1)**

### ▶️ Cek Kondisi Data Mentah

In [ ]:
print('=== 🔴 KONDISI SEBELUM PEMBERSIHAN (RAW DATA) ===')
print(f'Total Baris   : {len(df)}')
print(f'Total Kolom   : {df.shape[1]}')
print(f'Jumlah Duplikat: {df.duplicated().sum()}')
print('\nJumlah Missing Values per Kolom:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nStatistik Deskriptif:')
display(df.describe())

### ▶️ Visualisasi Distribusi Target & Hubungan Usia

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(x='heart_attack', data=df, palette='Set2')
plt.title('Distribusi Variabel Target (Heart Attack)')
plt.xlabel('Status (0: Aman, 1: Serangan Jantung)')

plt.subplot(1, 2, 2)
sns.boxplot(x='heart_attack', y='age', data=df, palette='Pastel1')
plt.title('Hubungan Usia vs Kejadian Serangan Jantung')
plt.xlabel('Status Serangan Jantung')
plt.ylabel('Usia (Tahun)')

plt.tight_layout()
plt.show()

---

## 🛠️ 5. Data Preparation

1. **Label Encoding 🧙‍♂️:** Konversi kolom kategorikal ke numerik.
2. **Feature Selection 🎯:** Memilih 9 fitur utama.
3. **Train-Test Split ✂️:** 80% latih, 20% uji dengan `stratify`.
4. **Feature Scaling 📏:** `StandardScaler` pada fitur numerik.

### ▶️ Eksekusi Data Preparation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_prepared = df.copy()

le = LabelEncoder()
for col in df_prepared.columns:
    if df_prepared[col].dtype == 'object':
        df_prepared[col] = le.fit_transform(df_prepared[col].astype(str))

fitur_pilihan = ['age', 'gender', 'hypertension', 'diabetes', 'cholesterol_level',
                 'obesity', 'smoking_status', 'dietary_habits', 'stress_level']

X = df_prepared[fitur_pilihan].fillna(0)
y = df_prepared['heart_attack']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Proses Data Preparation Selesai Sukses!')
print(f'X_train shape: {X_train_scaled.shape} | X_test shape: {X_test_scaled.shape}')

### 📝 Laporan Kualitas Data

| Aspek | Status | Keterangan |
|---|---|---|
| 👥 **Duplikat** | ✅ Bersih | Tidak ditemukan baris duplikat |
| ❓ **Missing Values** | 🛠️ Ditangani | `alcohol_consumption` tidak dipakai sebagai fitur |
| 📊 **Distribusi Target** | ⚠️ Imbalance | Kelas 0: 94.854 \| Kelas 1: 63.501 |
| 🔢 **Encoding** | ✅ Selesai | Semua kolom kategorikal dikonversi ke numerik |
| 📏 **Scaling** | ✅ Selesai | StandardScaler diterapkan pada fitur numerik |

---

## 🤖 6. Modeling

* **Logistic Regression:** Memetakan kombinasi linear ke fungsi sigmoid. 📉
* **XGBoost Classifier:** Ensemble decision trees dengan regularisasi formal. ⚡

### ▶️ Training Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

model_lr = LogisticRegression(random_state=42)
model_xgb = XGBClassifier(random_state=42, eval_metric='logloss')

model_lr.fit(X_train_scaled, y_train)
model_xgb.fit(X_train_scaled, y_train)

print('✅ Model Logistic Regression dan XGBoost Classifier berhasil dilatih!')

---

## 🏆 7. Evaluation

Metrik **Recall** menjadi prioritas utama untuk meminimalisir *False Negative*. 📊🔬

### ▶️ Classification Report & Confusion Matrix

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_lr = model_lr.predict(X_test_scaled)
y_pred_xgb = model_xgb.predict(X_test_scaled)

print('=================== REPORT LOGISTIC REGRESSION ===================')
print(classification_report(y_test, y_pred_lr))

print('\n==================== REPORT XGBOOST CLASSIFIER ===================')
print(classification_report(y_test, y_pred_xgb))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues', ax=ax[0])
ax[0].set_title('Confusion Matrix - Logistic Regression')
ax[0].set_xlabel('Prediksi')
ax[0].set_ylabel('Kenyataan Asli')

sns.heatmap(confusion_matrix(y_test, y_pred_xgb), annot=True, fmt='d', cmap='Greens', ax=ax[1])
ax[1].set_title('Confusion Matrix - XGBoost Classifier')
ax[1].set_xlabel('Prediksi')
ax[1].set_ylabel('Kenyataan Asli')

plt.tight_layout()
plt.show()

### ▶️ Contoh Hasil Prediksi vs Jawaban Asli

In [ ]:
target_name = ['Risiko Rendah', 'Risiko Tinggi']

contoh_xgb = X_test.copy().reset_index(drop=True)
contoh_xgb['Diagnosis Asli']     = [target_name[l] for l in y_test.values]
contoh_xgb['Diagnosis Prediksi'] = [target_name[l] for l in y_pred_xgb]
contoh_xgb['Benar?'] = ['✅' if a == p else '❌' for a, p in zip(y_test.values, y_pred_xgb)]

print('10 Contoh Hasil Prediksi XGBoost:')
print(contoh_xgb[['Diagnosis Asli', 'Diagnosis Prediksi', 'Benar?']].head(10).to_string())

### 📊 Hasil Performa Model

| Metrik Evaluasi | Logistic Regression 📉 | XGBoost Classifier ⚡ |
| :--- | :---: | :---: |
| **Akurasi Global** | 0.70 (70%) | **0.70 (70%)** |
| **Precision (Kelas 1)** | 0.65 | **0.65** |
| **Recall (Kelas 1)** | 0.51 | **0.54** |
| **F1-Score (Kelas 1)** | 0.57 | **0.59** |

### 🔍 Analisis Precision vs Recall

- **Precision 0.65**: Dari semua pasien yang diprediksi berisiko tinggi, 65% memang benar-benar berisiko.
- **Recall LR (0.51)**: Logistic Regression hanya mendeteksi 51% pasien berisiko — **49% tidak terdeteksi**.
- **Recall XGBoost (0.54)**: XGBoost mendeteksi 54% — **lebih baik 3% dibanding LR**.

### ⚠️ Analisis Ketidakseimbangan Data

Dataset memiliki distribusi tidak seimbang: kelas 0 = **94.854** vs kelas 1 = **63.501**. Kondisi ini membuat model cenderung lebih mudah memprediksi kelas mayoritas sehingga Recall kelas 1 menjadi lebih rendah.

---

## 🚀 8. Deployment

Model terbaik (**XGBoost**) diekspor ke `.pkl` menggunakan `joblib` dan dideploy di **Hugging Face Spaces** dengan **Gradio**. ⚙️🌐

### ▶️ Simpan dan Download Model

In [ ]:
import joblib
from google.colab import files

joblib.dump(model_xgb, 'model_SerJan_xgb.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('✅ Berhasil membuat file .pkl!')
print('⬇️ Memulai pengunduhan file ke komputer...')
files.download('model_SerJan_xgb.pkl')
files.download('scaler.pkl')

---

## ✅ 9. Kesimpulan

1. **Faktor Risiko Utama:** Fitur `cholesterol_level`, `hypertension`, `diabetes`, dan `stress_level` memiliki korelasi signifikan terhadap serangan jantung.

2. **Perbandingan Model:** **XGBoost Classifier** lebih unggul dengan **Recall 0.54** vs Logistic Regression **0.51** — lebih sedikit pasien berisiko yang tidak terdeteksi (*False Negative*).

3. **Ketidakseimbangan Data:** Distribusi 60:40 menjadi tantangan utama. Teknik SMOTE atau *class weighting* dapat dipertimbangkan untuk iterasi berikutnya.

4. **Model Terpilih:** **XGBoost Classifier** dipilih sebagai model final karena Recall lebih tinggi dan lebih andal sebagai instrumen skrining preventif. 🎉